# grad-accumulate-on-leaf — ex3: gradient accumulation across micro-batches — divide loss by n, zero only after step

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grad-accumulate-on-leaf`. Running the final beacon cell reports progress against the `Backprop: Grad accumulate on leaf` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Grad accumulate on leaf` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grad-accumulate-on-leaf`** (exercise 3). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grad-accumulate-on-leaf"
DD_SUBTOPIC = "Backprop: Grad accumulate on leaf"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Gradient accumulation across micro-batches — accumulate on PURPOSE

Ex1 implemented the per-touch accumulate. Ex2 added `zero_grad` to clear between training steps. The third facet flips the script: INTENTIONAL multi-step accumulation across MICRO-BATCHES, where the absence of zero_grad is the feature, not the bug.

```python
# Effective batch_size = micro_bs * n_microbatches, with constant memory.
for microbatch in batches:
    loss = compute_loss(model, microbatch) / n_microbatches
    backward(loss)        # adds to p.grad for each param
# After the loop, p.grad is the average over the full effective batch.
optimizer_step(model, lr)
zero_grad(model.parameters())  # clear ONLY after the optimizer step
```

**Why divide each micro-loss by `n_microbatches`.** Without the division, summing N micro-losses gives N x the desired loss — the gradient is N x too large. Dividing inside the loop is mathematically equivalent to averaging the full batch's loss, but you only ever hold one micro-batch's activations in memory.

**Why no zero_grad between micro-steps.** That's the whole point — the accumulation IS the simulated big-batch gradient. Calling zero_grad between micro-batches would defeat the purpose and only the last micro-batch's gradient would inform the update.

**Why this is its own pattern.** Gradient accumulation is how you train 64-batch-size models on a GPU that only fits 8. The pattern is load-bearing for every LLM training run since GPT-2 — and it's fundamentally the same `accumulate_grad` from ex1, just deliberately called across multiple forward/backward passes before zeroing.

### Exercise 3 — gradient accumulation across micro-batches — divide loss by n, zero only after step

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Apply
> LO: Apply the micro-batch gradient-accumulation pattern: divide each micro-batch's loss by the micro-batch count, accumulate gradients across micro-batches WITHOUT zero_grad in between, and zero only after the (simulated) optimizer step.
> Keywords: microbatch, accumulation, effective-batch, loss-scale, training-loop
> ```

**KCs targeted:** `intentional-accumulation-across-microbatches`, `loss-divide-by-microbatch-count`

Implement `ex3_train_one_effective_batch(param, microbatch_grads, lr)`. Simulate one optimizer step over an EFFECTIVE batch that's split into multiple micro-batches.

Inputs:
- `param`: a `MiniTensor` with `requires_grad=True`, starting with `param.grad = None`. This is the parameter to update.
- `microbatch_grads`: a list of `torch.Tensor`s — one gradient per micro-batch, all the same shape as `param.array`. Each tensor is what a backward pass over that micro-batch would have produced (NOT yet divided).
- `lr`: float, learning rate.

Behavior:

1. **For each micro-batch gradient `g`** in `microbatch_grads`:
   - Compute `g_scaled = g / len(microbatch_grads)` — emulates dividing the per-microbatch loss by N (the canonical gradient-accumulation pattern).
   - Accumulate into `param.grad` using the ex1 pattern: set on first touch (when `param.grad is None`), add otherwise — and ALWAYS rebind (`param.grad = param.grad + g_scaled`), NOT in-place `+=`.
   - DO NOT zero `param.grad` between micro-batches — the accumulation IS the simulated big-batch gradient.
2. **After the loop, simulate one optimizer step**:
   - `param.array = param.array - lr * param.grad`.
3. **Then zero the grad** (set `param.grad = None`) — this is the convention from ex2.
4. **Return** the updated `param` (same MiniTensor object).

Constraints:
- Accumulation MUST use rebinding `+`, not in-place `+=` (so external references to the old grad tensor stay intact).
- `param.grad` MUST be `None` after the function returns (post-step zero).
- `param.array` MUST reflect the AVERAGE gradient applied with `lr`, not the sum.

In [ ]:
def ex3_train_one_effective_batch(param: MiniTensor, microbatch_grads: list, lr: float):
    """One optimizer step over an effective batch split into N micro-batches."""
    raise NotImplementedError()


def _test_ex3():
    def _test_ex3():
        # === Single micro-batch (degenerate case — equivalent to one normal step) ===
        p = MiniTensor(t.tensor([10.0, 10.0]), requires_grad=True)
        g1 = t.tensor([2.0, 4.0])
        result = ex3_train_one_effective_batch(p, [g1], lr=0.1)
        assert result is p, 'must return same MiniTensor object'
        # 1 micro-batch: scaled grad = g1 / 1 = g1; param -= 0.1 * g1 → [9.8, 9.6]
        assert t.allclose(p.array, t.tensor([9.8, 9.6])), (
            f'single-microbatch update wrong: {p.array}'
        )
        assert p.grad is None, f'grad must be zeroed after step; got {p.grad}'

        # === Four equal micro-batches — should equal one big step with the average gradient ===
        p = MiniTensor(t.tensor([10.0]), requires_grad=True)
        grads = [t.tensor([4.0]), t.tensor([4.0]), t.tensor([4.0]), t.tensor([4.0])]
        ex3_train_one_effective_batch(p, grads, lr=0.1)
        # Each grad / 4 = 1.0, accumulated 4 times = 4.0; lr*4.0 = 0.4 → 10.0 - 0.4 = 9.6
        assert t.allclose(p.array, t.tensor([9.6])), (
            f'4x identical grads should give a 4.0 average; got param={p.array}'
        )
        assert p.grad is None

        # === Non-uniform micro-batches: gradient is the AVERAGE, not the sum ===
        p = MiniTensor(t.tensor([0.0]), requires_grad=True)
        grads = [t.tensor([10.0]), t.tensor([20.0])]   # average = 15
        ex3_train_one_effective_batch(p, grads, lr=1.0)
        # accumulated = 10/2 + 20/2 = 15; param = 0 - 1.0*15 = -15
        assert t.allclose(p.array, t.tensor([-15.0])), (
            f'should apply AVERAGE gradient, got {p.array}'
        )

        # === Equivalence: micro-batched matches one big batch with the averaged grad ===
        # Reference: one-shot step with avg gradient.
        p_ref = MiniTensor(t.tensor([5.0, 5.0]), requires_grad=True)
        g_a, g_b, g_c = t.tensor([1.0, 0.0]), t.tensor([0.0, 1.0]), t.tensor([1.0, 1.0])
        avg = (g_a + g_b + g_c) / 3
        p_ref.array = p_ref.array - 0.5 * avg

        p = MiniTensor(t.tensor([5.0, 5.0]), requires_grad=True)
        ex3_train_one_effective_batch(p, [g_a, g_b, g_c], lr=0.5)
        assert t.allclose(p.array, p_ref.array, atol=1e-7), (
            f'microbatched != big-batch equivalent: {p.array} vs {p_ref.array}'
        )

        # === Rebinding semantics: no in-place += during accumulation ===
        # If we hold a reference to the grad mid-loop, it must NOT change underneath us.
        # We can't easily intercept mid-loop, so instead we verify final-grad-rebinding
        # by checking that p.grad has been set to None (which can't happen if the
        # earlier grads were in-place mutating the same buffer — see ex1).
        p = MiniTensor(t.tensor([0.0]), requires_grad=True)
        ex3_train_one_effective_batch(p, [t.tensor([1.0]), t.tensor([2.0])], lr=0.0)
        # lr=0 → param unchanged but step still runs; grad still zeroed at end.
        assert p.array.item() == 0.0
        assert p.grad is None, 'zero_grad after step is mandatory'

        # === Sequential calls: second call works fine after first cleared grad ===
        p = MiniTensor(t.tensor([10.0]), requires_grad=True)
        ex3_train_one_effective_batch(p, [t.tensor([2.0]), t.tensor([2.0])], lr=0.1)
        after_first = p.array.clone()
        ex3_train_one_effective_batch(p, [t.tensor([4.0]), t.tensor([4.0])], lr=0.1)
        # First call: avg grad 2.0, step 0.2 → 9.8
        # Second call: avg grad 4.0, step 0.4 → 9.4
        assert t.allclose(after_first, t.tensor([9.8]))
        assert t.allclose(p.array, t.tensor([9.4])), (
            f'consecutive accumulation steps must not leak grads: {p.array}'
        )

        # === Two micro-batches with zero-sum gradients → no parameter update ===
        p = MiniTensor(t.tensor([100.0]), requires_grad=True)
        ex3_train_one_effective_batch(p, [t.tensor([1.0]), t.tensor([-1.0])], lr=10.0)
        # avg = 0 → param unchanged.
        assert t.allclose(p.array, t.tensor([100.0])), (
            f'zero-sum micro-grads must not move param: {p.array}'
        )
        print('ex3 ✓')

    _test_ex3()
    _dd_passed.add('ex3')
    print("ex3 ✓")

_test_ex3()

<details><summary>Solution</summary>

```python
def ex3_train_one_effective_batch(param, microbatch_grads, lr):
    n = len(microbatch_grads)
    # Accumulate per-microbatch gradients (each scaled by 1/n) into param.grad.
    # Mirrors the canonical pattern: divide micro-loss by n inside the loop;
    # NO zero_grad between micro-batches — the accumulation IS the big-batch grad.
    for g in microbatch_grads:
        g_scaled = g / n
        if param.grad is None:
            param.grad = g_scaled
        else:
            # Rebind, not in-place — external references stay valid.
            param.grad = param.grad + g_scaled
    # Simulated optimizer step: param.array -= lr * param.grad (leaf in-place is safe).
    param.array = param.array - lr * param.grad
    # Zero the grad NOW (after the step), not between micro-batches.
    param.grad = None
    return param
```

**The division-by-N is the load-bearing detail.** Without it, accumulating N micro-batches gives an N-times-too-large gradient — your effective lr is silently 4x or 16x what you set. Some papers report this bug as 'the model trained fine but used the wrong lr'.

**Why zero_grad goes AFTER the optimizer step, not between micro-batches.** Between micro-batches we WANT the gradients to stack — that's the whole pattern. Only after the step has consumed the accumulated gradient do we clear it for the next effective batch.

**Rebind, never `+=`.** Ex1 covered this for safety against external references; here it matters for the same reason — if a logging hook snapshots `param.grad` mid-loop, in-place mutation would change the snapshot retroactively.

**Why this is the third facet.** Ex1 was the per-touch primitive. Ex2 was the inter-step boundary (`zero_grad`). Ex3 is the intra-step intentional accumulation — a deliberate use of the ex1 mechanism that ex2's discipline (no leaked grads between STEPS) doesn't preclude. The three exercises together cover the full lifecycle of `param.grad`.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex3',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()